In [1]:
pip install torch numpy tslearn pandas mantis-tsfm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 80.9 MB/s  0:00:09m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 100.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 87.2 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 89.0 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 22.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 73.3 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 81.6 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 87.2 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 99.0 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 8

# Project DL4TS
### By Alice Lataste, Thomas Roussaux, Spyridon Paipetis and Soline Mignot

The project aims to do some time series classification on the LSST dataset from the tslearn library.

In [2]:
from sklearn.metrics import accuracy_score
#from tslearn.shapelets import LearningShapelets
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tslearn.datasets import UCR_UEA_datasets

## Fetching the dataset and normalizing it

In [3]:
from tslearn.datasets import UCR_UEA_datasets
# Load the LSST dataset from UEA archive
ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

In [4]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(len(set(y_train)))

(2459, 36, 6)
(2466, 36, 6)
(2459,)
14


In [5]:
# Normalization
scaler = TimeSeriesScalerMeanVariance()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# encode labels
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# conversion torch
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32)

## Part 1 : Baseline model: CNN

### Step 1 : creating the class

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNNBaseline(nn.Module):
    def __init__(self, n_channels=6, n_classes=14):
        super().__init__()
        
        self.conv1 = nn.Conv1d(in_channels=n_channels, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        
        self.conv3 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        
        self.pool = nn.AdaptiveAvgPool1d(1)
        
        self.fc1 = nn.Linear(128, 64)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, n_classes)

    def forward(self, x):
        # x: (batch, 36, 6) -> (batch, 6, 36)
        x = x.permute(0, 2, 1)
        
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        
        x = self.pool(x).squeeze(-1)   # (batch, 128)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

### Step 2 : Model initialization and training


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = CNNBaseline(n_channels=6, n_classes=14).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

cuda


In [8]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * X_batch.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)
    
    return total_loss / total, correct / total

### Step 3 : Evaluation

In [9]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            total_loss += loss.item() * X_batch.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    
    return total_loss / total, correct / total

### Step 4 : Run the model

In [10]:
n_epochs = 100

for epoch in range(n_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    
    print(f"Epoch {epoch+1:02d}/{n_epochs} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

Epoch 01/100 | Train Loss: 2.5499 | Train Acc: 0.2338 | Test Loss: 2.4480 | Test Acc: 0.3159
Epoch 02/100 | Train Loss: 2.3548 | Train Acc: 0.3266 | Test Loss: 2.2319 | Test Acc: 0.3155
Epoch 03/100 | Train Loss: 2.1542 | Train Acc: 0.3314 | Test Loss: 2.0418 | Test Acc: 0.3236
Epoch 04/100 | Train Loss: 2.0139 | Train Acc: 0.3660 | Test Loss: 1.9098 | Test Acc: 0.3812
Epoch 05/100 | Train Loss: 1.9070 | Train Acc: 0.4099 | Test Loss: 1.8084 | Test Acc: 0.4461
Epoch 06/100 | Train Loss: 1.8172 | Train Acc: 0.4294 | Test Loss: 1.7232 | Test Acc: 0.4984
Epoch 07/100 | Train Loss: 1.7453 | Train Acc: 0.4477 | Test Loss: 1.6511 | Test Acc: 0.5316
Epoch 08/100 | Train Loss: 1.6787 | Train Acc: 0.4839 | Test Loss: 1.5912 | Test Acc: 0.5393
Epoch 09/100 | Train Loss: 1.6236 | Train Acc: 0.5018 | Test Loss: 1.5321 | Test Acc: 0.5552
Epoch 10/100 | Train Loss: 1.5786 | Train Acc: 0.5075 | Test Loss: 1.4945 | Test Acc: 0.5633
Epoch 11/100 | Train Loss: 1.5484 | Train Acc: 0.5132 | Test Loss: 1.4

Epoch 65/100 | Train Loss: 0.9740 | Train Acc: 0.6865 | Test Loss: 1.1322 | Test Acc: 0.6415
Epoch 66/100 | Train Loss: 0.9564 | Train Acc: 0.6909 | Test Loss: 1.1319 | Test Acc: 0.6403
Epoch 67/100 | Train Loss: 0.9645 | Train Acc: 0.6922 | Test Loss: 1.1268 | Test Acc: 0.6427
Epoch 68/100 | Train Loss: 0.9524 | Train Acc: 0.6934 | Test Loss: 1.1293 | Test Acc: 0.6460
Epoch 69/100 | Train Loss: 0.9548 | Train Acc: 0.7003 | Test Loss: 1.1347 | Test Acc: 0.6379
Epoch 70/100 | Train Loss: 0.9517 | Train Acc: 0.6889 | Test Loss: 1.1309 | Test Acc: 0.6415
Epoch 71/100 | Train Loss: 0.9474 | Train Acc: 0.6913 | Test Loss: 1.1420 | Test Acc: 0.6411
Epoch 72/100 | Train Loss: 0.9328 | Train Acc: 0.6946 | Test Loss: 1.1352 | Test Acc: 0.6367
Epoch 73/100 | Train Loss: 0.9291 | Train Acc: 0.6987 | Test Loss: 1.1315 | Test Acc: 0.6444
Epoch 74/100 | Train Loss: 0.9181 | Train Acc: 0.7096 | Test Loss: 1.1327 | Test Acc: 0.6379
Epoch 75/100 | Train Loss: 0.9155 | Train Acc: 0.7031 | Test Loss: 1.1

## Part 2 : Strong competitor

For this second part of the project, we have chosen Setting 1 : Adapt a foundation model. 

In [19]:
import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)  # (max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]


class TimeSeriesTransformerClassifier(nn.Module):
    def __init__(
        self,
        input_dim=6,
        seq_len=36,
        d_model=64,
        nhead=4,
        num_layers=3,
        dim_feedforward=128,
        dropout=0.1,
        n_classes=14
    ):
        super().__init__()

        self.input_projection = nn.Linear(input_dim, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len=seq_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        # x: (batch, 36, 6)
        x = self.input_projection(x)              # (batch, 36, d_model)
        x = self.positional_encoding(x)           # (batch, 36, d_model)
        x = self.transformer(x)                   # (batch, 36, d_model)
        x = x.mean(dim=1)                         # global average pooling over time
        x = self.dropout(x)
        x = self.classifier(x)                    # (batch, n_classes)
        return x

## Step 2: Initialisation modèle

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

transformer_model = TimeSeriesTransformerClassifier(
    input_dim=6,
    seq_len=36,
    d_model=64,
    nhead=4,
    num_layers=3,
    dim_feedforward=128,
    dropout=0.1,
    n_classes=14
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(transformer_model.parameters(), lr=1e-4)

cpu


## Lancement entraînement

In [23]:
n_epochs = 100

for epoch in range(n_epochs):
    train_loss, train_acc = train_one_epoch(
        transformer_model, train_loader, criterion, optimizer, device
    )
    test_loss, test_acc = evaluate(
        transformer_model, test_loader, criterion, device
    )

    print(
        f"Epoch {epoch+1:02d}/{n_epochs} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}"
    )

Epoch 01/100 | Train Loss: 1.2471 | Train Acc: 0.5913 | Test Loss: 1.3000 | Test Acc: 0.5815
Epoch 02/100 | Train Loss: 1.2325 | Train Acc: 0.6035 | Test Loss: 1.3023 | Test Acc: 0.5843
Epoch 03/100 | Train Loss: 1.2246 | Train Acc: 0.5937 | Test Loss: 1.2943 | Test Acc: 0.5766
Epoch 04/100 | Train Loss: 1.2126 | Train Acc: 0.6002 | Test Loss: 1.2869 | Test Acc: 0.5896
Epoch 05/100 | Train Loss: 1.2072 | Train Acc: 0.5986 | Test Loss: 1.2791 | Test Acc: 0.5884
Epoch 06/100 | Train Loss: 1.1946 | Train Acc: 0.6059 | Test Loss: 1.2747 | Test Acc: 0.5908
Epoch 07/100 | Train Loss: 1.1821 | Train Acc: 0.6080 | Test Loss: 1.2833 | Test Acc: 0.5880
Epoch 08/100 | Train Loss: 1.1955 | Train Acc: 0.5986 | Test Loss: 1.2754 | Test Acc: 0.5929
Epoch 09/100 | Train Loss: 1.1780 | Train Acc: 0.6169 | Test Loss: 1.2694 | Test Acc: 0.5945
Epoch 10/100 | Train Loss: 1.1741 | Train Acc: 0.6141 | Test Loss: 1.2766 | Test Acc: 0.5864
Epoch 11/100 | Train Loss: 1.1550 | Train Acc: 0.6198 | Test Loss: 1.2